In [1]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch import Tensor
import pandas as pd
from torchmetrics.classification import MultilabelAveragePrecision
import torch, dotenv, os
from utils import transform_train, transform_test
from model import Tagger
from torch.utils.tensorboard import SummaryWriter

dotenv.load_dotenv()
token = os.getenv('HF_TOKEN')

pd.set_option('display.float_format', '{:.3f}'.format)

In [2]:
to_predict = (
    'female', 'male',
    'unicorn', 'pegasus', 'earth pony', 'alicorn',
    'simple background', 'monochrome',
    'clothes', 'wings', 'horn', 'chest fluff', 'ear fluff', 'hat', 'jewelry', 'foal',
    'looking at you', 'smiling', 'open mouth', 'blushing', 'sitting', 'raised hoof', 'eyes closed',
    'twilight sparkle', 'fluttershy', 'rainbow dash', 'pinkie pie', 'rarity', 'applejack', 'princess celestia', 'princess luna',
    'socks', 'text', 'tongue out', 'lying down'
)

In [3]:
ds = load_dataset('Brambles/Ponies', token=token)

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/59 [00:00<?, ?it/s]

In [4]:
epochs = 10
batch_size = 168
lr = 0.003
weight_decay = 0.001
temp_alpha = 1.0
mix_alpha = 0.3
gamma = 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [5]:
def t_train(batch):
    return {
        'images': torch.stack([transform_train(img) for img in batch['image']]),
        'tags': torch.tensor(batch['tags'], dtype=torch.float32)
    }

def t_val(batch):
    return {
        'images': torch.stack([transform_test(img) for img in batch['image']]),
        'tags': torch.tensor(batch['tags'], dtype=torch.float32)
    }

In [6]:
dl_train = DataLoader(
    ds['train'].with_transform(t_train),
    batch_size=batch_size,
    pin_memory=True,
    num_workers=4,
    persistent_workers=True,
    shuffle=True,
    drop_last=True
)

dl_validation = DataLoader(
    ds['validation'].with_transform(t_val),
    batch_size=batch_size,
    pin_memory=True,
    num_workers=3,
    persistent_workers=True,
)

len(dl_train)

1061

In [7]:
model = Tagger().to(device)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

In [8]:
optim = AdamW([
        { 'params': model.backbone.parameters(), 'lr': lr * 0.05 },
        { 'params': model.classifier.parameters(), 'lr': lr },
    ],
    weight_decay=weight_decay
)

scheduler = CosineAnnealingLR(optim, epochs)

In [9]:
from loss import VPULoss, MixupLoss

vpu_loss = VPULoss(temp_alpha, gamma)
mixup_loss = MixupLoss(mix_alpha)

In [10]:
f_mAP = MultilabelAveragePrecision(len(to_predict))

def evaluate(epoch, best_mAP):
    model.eval()
    preds = torch.zeros((len(ds['validation']), len(to_predict)))
    val_loss = 0

    with torch.no_grad():
        for i, batch in enumerate(dl_validation):
            X: torch.Tensor = batch['images'].to(device, non_blocking=True)
            y: torch.Tensor = batch['tags'].to(device, non_blocking=True)

            logits: torch.Tensor = model(X)
            val_loss += vpu_loss(logits, y).sum()

            preds[i * batch_size:min((i + 1) * batch_size, len(ds['validation']))] = logits.sigmoid()

        f_mAP.reset()
        mAP: Tensor = f_mAP(preds, torch.as_tensor(ds['validation']['tags']))

        if mAP > best_mAP:
            best_mAP = mAP
            prefix = 'Best-'
        else:
            prefix = ''

        if epoch >= 4:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optim.state_dict(),
            }, f'outputs/{prefix}E{epoch}.pt')

    model.train()
    return val_loss / len(dl_validation), mAP, best_mAP

In [11]:
writer = SummaryWriter()
validations_per_epoch = 10
best_mAP = 0

In [12]:
def train_epoch(epoch):
    global best_mAP

    model.train()
    step = (epoch - 1) * len(dl_train)

    for i, batch in enumerate(dl_train, 1):
        X: torch.Tensor = batch['images'].to(device, non_blocking=True)
        y: torch.Tensor = batch['tags'].to(device, non_blocking=True)

        with torch.autocast('cuda', torch.bfloat16):
            logits: Tensor = model(X).float()

        preds = logits.sigmoid()

        loss = vpu_loss(logits, y).sum()

        # No mixup on the last one
        if epoch != epochs:
            loss += mixup_loss(X, preds, y, model)

        loss_delta = loss.item()

        loss.backward()
        optim.step()
        optim.zero_grad()

        step += 1

        if i % 5 == 0:
            lrs = [f'{param_group['lr']:.6f}' for param_group in optim.param_groups]
            writer.add_scalar('Loss/train', loss_delta, step)
            writer.add_scalar('Conf', preds.sum() / preds.numel(), step)

            print(f'{' '*40}\r{epoch}/{epochs}: %{i * 100 / len(dl_train):.2f} lrs: {lrs}', end='\r')

        if i % (len(dl_train) // validations_per_epoch) == 0:
            val_loss, mAP, best_mAP = evaluate(epoch, best_mAP)
            writer.add_scalar('Loss/val', val_loss, step)
            writer.add_scalar('mAP/val', mAP, step)

    scheduler.step()

In [13]:
for epoch in range(1, epochs + 1):
    model.freeze_backbone(epoch == 1)
    train_epoch(epoch)
